# Three-Class Image Classification Notebook

This notebook covers four tasks using the output labels `babby`, `flower`, and `elephant`:

1. Create your own PyTorch CNN.
2. Create your own TensorFlow CNN.
3. Do transfer learning with VGG19.
4. Bring a new image and classify it.

## Dataset Structure

Create a dataset folder with one subfolder per class:

```
dataset/
  babby/
    img1.jpg
    img2.jpg
  flower/
    img1.jpg
  elephant/
    img1.jpg
```

Update `DATASET_DIR` below before running the training cells.

In [ ]:
# If your environment is missing packages, uncomment and run the next line.
# %pip install torch torchvision tensorflow pillow matplotlib scikit-learn

from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import tensorflow as tf

CLASS_NAMES = ["babby", "flower", "elephant"]
DATASET_DIR = Path("dataset")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 10
IMG_SIZE = (128, 128)
# AUTOTUNE is defined here so every section can use it without ordering issues
AUTOTUNE = tf.data.AUTOTUNE

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch device: {device}")
print(f"Dataset directory exists: {DATASET_DIR.exists()}")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_samples(dataset_dir, class_names):
    samples = []
    for label_index, class_name in enumerate(class_names):
        class_dir = dataset_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                samples.append((image_path, label_index))
    if not samples:
        raise ValueError(f"No images found in {dataset_dir}")
    return samples

def split_samples(samples, test_size=0.2, val_size=0.1, random_state=42):
    file_paths = [str(path) for path, _ in samples]
    labels = [label for _, label in samples]

    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        file_paths,
        labels,
        test_size=test_size + val_size,
        stratify=labels,
        random_state=random_state,
    )

    relative_val_size = val_size / (test_size + val_size)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths,
        temp_labels,
        test_size=1 - relative_val_size,
        stratify=temp_labels,
        random_state=random_state,
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))
    test_samples = list(zip(test_paths, test_labels))
    return train_samples, val_samples, test_samples

def show_random_samples(samples, class_names, num_images=6):
    chosen = random.sample(samples, k=min(num_images, len(samples)))
    plt.figure(figsize=(12, 6))
    for index, (image_path, label_index) in enumerate(chosen, start=1):
        image = Image.open(image_path).convert("RGB")
        plt.subplot(2, 3, index)
        plt.imshow(image)
        plt.title(class_names[label_index])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

all_samples = collect_samples(DATASET_DIR, CLASS_NAMES)
train_samples, val_samples, test_samples = split_samples(all_samples, test_size=0.2, val_size=0.1, random_state=RANDOM_STATE)

print(f"Total images: {len(all_samples)}")
print(f"Train images: {len(train_samples)}")
print(f"Validation images: {len(val_samples)}")
print(f"Test images: {len(test_samples)}")
show_random_samples(all_samples, CLASS_NAMES)

## 1. Create Your Own PyTorch CNN

Run the next cell to build, train, evaluate, and save a custom PyTorch CNN.

In [ ]:
class PyTorchImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


# Training augmentation: flip, small rotation, colour jitter
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset_pt = PyTorchImageDataset(train_samples, transform=train_transform)
val_dataset_pt   = PyTorchImageDataset(val_samples,   transform=eval_transform)
test_dataset_pt  = PyTorchImageDataset(test_samples,  transform=eval_transform)

train_loader_pt = DataLoader(train_dataset_pt, batch_size=BATCH_SIZE, shuffle=True)
val_loader_pt   = DataLoader(val_dataset_pt,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_pt  = DataLoader(test_dataset_pt,  batch_size=BATCH_SIZE, shuffle=False)


class CustomPyTorchCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # After 3 x MaxPool2d(2): 128x128 → 16x16
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.features(inputs))


def evaluate_pytorch(model, data_loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct_predictions, total_samples = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct_predictions += (outputs.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)
    return total_loss / total_samples, correct_predictions / total_samples


pytorch_model = CustomPyTorchCNN(num_classes=len(CLASS_NAMES)).to(device)
criterion_pt  = nn.CrossEntropyLoss()
optimizer_pt  = torch.optim.Adam(pytorch_model.parameters(), lr=1e-3)

pt_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    pytorch_model.train()
    running_loss, running_correct, total_train = 0.0, 0, 0

    for images, labels in train_loader_pt:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_pt.zero_grad()
        outputs = pytorch_model(images)
        loss = criterion_pt(outputs, labels)
        loss.backward()
        optimizer_pt.step()

        running_loss    += loss.item() * images.size(0)
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train     += labels.size(0)

    train_loss, train_accuracy = running_loss / total_train, running_correct / total_train
    val_loss, val_accuracy = evaluate_pytorch(pytorch_model, val_loader_pt)

    pt_history["train_loss"].append(train_loss)
    pt_history["train_acc"].append(train_accuracy)
    pt_history["val_loss"].append(val_loss)
    pt_history["val_acc"].append(val_accuracy)
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  train_acc={train_accuracy:.4f}  val_loss={val_loss:.4f}  val_acc={val_accuracy:.4f}")

test_loss_pt, test_accuracy_pt = evaluate_pytorch(pytorch_model, test_loader_pt)
print(f"\nPyTorch test loss:    {test_loss_pt:.4f}")
print(f"PyTorch test accuracy: {test_accuracy_pt:.4f}")

# Training history plot
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, pt_history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, pt_history["val_loss"],   label="Val Loss")
axes[0].set_title("PyTorch CNN – Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs_range, pt_history["train_acc"], label="Train Acc")
axes[1].plot(epochs_range, pt_history["val_acc"],   label="Val Acc")
axes[1].set_title("PyTorch CNN – Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

pytorch_model_path = MODELS_DIR / "three_class_pytorch_cnn.pth"
torch.save({"model_state_dict": pytorch_model.state_dict(), "class_names": CLASS_NAMES}, pytorch_model_path)
print(f"Saved PyTorch model to {pytorch_model_path}")

In [ ]:
def predict_image_pytorch(image_path, model_path=pytorch_model_path):
    model = CustomPyTorchCNN(num_classes=len(CLASS_NAMES)).to(device)
    # weights_only=False required to load the class_names list alongside state_dict
    checkpoint = torch.load(str(model_path), map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    image = Image.open(str(image_path)).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    predicted_index = int(np.argmax(probabilities))
    plt.figure(figsize=(4, 4))
    plt.imshow(Image.open(str(image_path)).convert("RGB"))
    plt.axis("off")
    plt.title(f"PyTorch: {CLASS_NAMES[predicted_index]} ({probabilities[predicted_index]:.4f})")
    plt.show()
    return CLASS_NAMES[predicted_index], probabilities

NEW_IMAGE_PATH = Path("new_image.jpg")
# predict_image_pytorch(NEW_IMAGE_PATH)

## 2. Create Your Own TensorFlow CNN

Run the next cell to build, train, evaluate, and save a custom TensorFlow CNN.

In [ ]:
def load_tf_image(image_path, label, image_size):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def create_tf_dataset(samples, image_size, batch_size, shuffle=False, augment=False):
    paths  = [sample[0] for sample in samples]
    labels = [sample[1] for sample in samples]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(samples), seed=RANDOM_STATE)
    dataset = dataset.map(
        lambda path, label: load_tf_image(path, label, image_size),
        num_parallel_calls=AUTOTUNE,
    )
    if augment:
        dataset = dataset.map(
            lambda img, lbl: (tf.image.random_flip_left_right(img), lbl),
            num_parallel_calls=AUTOTUNE,
        )
    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset

train_ds_tf = create_tf_dataset(train_samples, IMG_SIZE, BATCH_SIZE, shuffle=True, augment=True)
val_ds_tf   = create_tf_dataset(val_samples,   IMG_SIZE, BATCH_SIZE)
test_ds_tf  = create_tf_dataset(test_samples,  IMG_SIZE, BATCH_SIZE)

tensorflow_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    # Block 1
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    # Block 2
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    # Block 3
    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    # Classifier
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
])

tensorflow_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
tensorflow_model.summary()

history_tf = tensorflow_model.fit(
    train_ds_tf,
    validation_data=val_ds_tf,
    epochs=EPOCHS,
    verbose=1,
)

test_loss_tf, test_accuracy_tf = tensorflow_model.evaluate(test_ds_tf, verbose=0)
print(f"\nTensorFlow test loss:     {test_loss_tf:.4f}")
print(f"TensorFlow test accuracy: {test_accuracy_tf:.4f}")

# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_tf.history["loss"],     label="Train Loss")
axes[0].plot(history_tf.history["val_loss"], label="Val Loss")
axes[0].set_title("TensorFlow CNN – Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history_tf.history["accuracy"],     label="Train Acc")
axes[1].plot(history_tf.history["val_accuracy"], label="Val Acc")
axes[1].set_title("TensorFlow CNN – Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

tensorflow_model_path = MODELS_DIR / "three_class_tensorflow_cnn.keras"
tensorflow_model.save(str(tensorflow_model_path))
print(f"Saved TensorFlow model to {tensorflow_model_path}")

In [ ]:
def predict_image_tensorflow(image_path, model_path=tensorflow_model_path):
    model = tf.keras.models.load_model(model_path)
    image = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    image_array = tf.keras.utils.img_to_array(image) / 255.0
    image_array = tf.expand_dims(image_array, axis=0)

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    plt.figure(figsize=(4, 4))
    plt.imshow(tf.keras.utils.load_img(image_path))
    plt.axis("off")
    plt.title(f"TensorFlow: {CLASS_NAMES[predicted_index]} ({probabilities[predicted_index]:.4f})")
    plt.show()
    return CLASS_NAMES[predicted_index], probabilities

# predict_image_tensorflow(NEW_IMAGE_PATH)

## 3. Transfer Learning with VGG19

This section uses VGG19 as a pretrained feature extractor and fine-tunes the last few layers.

In [ ]:
VGG_IMG_SIZE = (224, 224)

def load_vgg_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, VGG_IMG_SIZE)
    image = tf.keras.applications.vgg19.preprocess_input(tf.cast(image, tf.float32))
    return image, label

def create_vgg_dataset(samples, batch_size, shuffle=False):
    paths  = [sample[0] for sample in samples]
    labels = [sample[1] for sample in samples]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(samples), seed=RANDOM_STATE)
    dataset = dataset.map(load_vgg_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset

train_ds_vgg = create_vgg_dataset(train_samples, BATCH_SIZE, shuffle=True)
val_ds_vgg   = create_vgg_dataset(val_samples,   BATCH_SIZE)
test_ds_vgg  = create_vgg_dataset(test_samples,  BATCH_SIZE)

# Load VGG19 without the top classification head, freeze all base layers
base_model = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(VGG_IMG_SIZE[0], VGG_IMG_SIZE[1], 3),
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(VGG_IMG_SIZE[0], VGG_IMG_SIZE[1], 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(256, activation="relu")(x)
x       = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
vgg19_model = tf.keras.Model(inputs, outputs)

# ── Phase 1: Train the new head only ───────────────────────────────────────
print("── Phase 1: Train head only (VGG19 base frozen) ──")
vgg19_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_vgg_head = vgg19_model.fit(
    train_ds_vgg,
    validation_data=val_ds_vgg,
    epochs=5,
    verbose=1,
)

# ── Phase 2: Unfreeze last 4 VGG19 layers and fine-tune ────────────────────
print("\n── Phase 2: Fine-tune last 4 VGG19 layers ──")
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

vgg19_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_vgg_finetune = vgg19_model.fit(
    train_ds_vgg,
    validation_data=val_ds_vgg,
    epochs=3,
    verbose=1,
)

# Merge both phases for a continuous plot
all_loss    = history_vgg_head.history["loss"]        + history_vgg_finetune.history["loss"]
all_vloss   = history_vgg_head.history["val_loss"]    + history_vgg_finetune.history["val_loss"]
all_acc     = history_vgg_head.history["accuracy"]    + history_vgg_finetune.history["accuracy"]
all_vacc    = history_vgg_head.history["val_accuracy"]+ history_vgg_finetune.history["val_accuracy"]
all_epochs  = range(1, len(all_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(all_epochs, all_loss,  label="Train Loss")
axes[0].plot(all_epochs, all_vloss, label="Val Loss")
axes[0].axvline(5.5, color="gray", linestyle="--", label="Fine-tune start")
axes[0].set_title("VGG19 Transfer Learning – Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(all_epochs, all_acc,  label="Train Acc")
axes[1].plot(all_epochs, all_vacc, label="Val Acc")
axes[1].axvline(5.5, color="gray", linestyle="--", label="Fine-tune start")
axes[1].set_title("VGG19 Transfer Learning – Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

test_loss_vgg, test_accuracy_vgg = vgg19_model.evaluate(test_ds_vgg, verbose=0)
print(f"\nVGG19 test loss:     {test_loss_vgg:.4f}")
print(f"VGG19 test accuracy: {test_accuracy_vgg:.4f}")

vgg19_model_path = MODELS_DIR / "three_class_vgg19.keras"
vgg19_model.save(str(vgg19_model_path))
print(f"Saved VGG19 model to {vgg19_model_path}")

In [ ]:
def predict_image_vgg19(image_path, model_path=vgg19_model_path):
    model = tf.keras.models.load_model(model_path)
    image = tf.keras.utils.load_img(image_path, target_size=VGG_IMG_SIZE)
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, axis=0)
    image_array = tf.keras.applications.vgg19.preprocess_input(image_array)

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    plt.figure(figsize=(4, 4))
    plt.imshow(tf.keras.utils.load_img(image_path))
    plt.axis("off")
    plt.title(f"VGG19: {CLASS_NAMES[predicted_index]} ({probabilities[predicted_index]:.4f})")
    plt.show()
    return CLASS_NAMES[predicted_index], probabilities

# predict_image_vgg19(NEW_IMAGE_PATH)

## 4. Predict a New Image with All Three Models

Set `NEW_IMAGE_PATH` to your own JPG/PNG and run the cell below.  
The function loads each saved model and shows a side-by-side comparison with a confidence table.

In [ ]:
def classify_new_image(image_path):
    """Run all three trained models on a new image and display a side-by-side comparison."""
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    # ── PyTorch prediction ──────────────────────────────────────────────────
    pt_model = CustomPyTorchCNN(num_classes=len(CLASS_NAMES)).to(device)
    ckpt = torch.load(str(pytorch_model_path), map_location=device, weights_only=False)
    pt_model.load_state_dict(ckpt["model_state_dict"])
    pt_model.eval()
    tensor = eval_transform(Image.open(str(image_path)).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        pt_probs = torch.softmax(pt_model(tensor), dim=1)[0].cpu().numpy()
    pt_idx = int(np.argmax(pt_probs))

    # ── TensorFlow CNN prediction ───────────────────────────────────────────
    tf_model = tf.keras.models.load_model(str(tensorflow_model_path))
    img_tf   = tf.keras.utils.img_to_array(tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)) / 255.0
    tf_probs = tf_model.predict(tf.expand_dims(img_tf, 0), verbose=0)[0]
    tf_idx   = int(np.argmax(tf_probs))

    # ── VGG19 prediction ────────────────────────────────────────────────────
    vgg_model = tf.keras.models.load_model(str(vgg19_model_path))
    img_vgg   = tf.keras.utils.img_to_array(tf.keras.utils.load_img(image_path, target_size=VGG_IMG_SIZE))
    img_vgg   = tf.keras.applications.vgg19.preprocess_input(tf.expand_dims(img_vgg, 0))
    vgg_probs = vgg_model.predict(img_vgg, verbose=0)[0]
    vgg_idx   = int(np.argmax(vgg_probs))

    # ── Side-by-side display ─────────────────────────────────────────────────
    original = Image.open(str(image_path)).convert("RGB")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, model_name, idx, probs in zip(
        axes,
        ["PyTorch CNN", "TensorFlow CNN", "VGG19 Transfer"],
        [pt_idx, tf_idx, vgg_idx],
        [pt_probs, tf_probs, vgg_probs],
    ):
        ax.imshow(original)
        ax.axis("off")
        ax.set_title(f"{model_name}\n{CLASS_NAMES[idx]}  ({probs[idx]:.4f})", fontsize=12)
    plt.suptitle(f"Input: {image_path.name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ── Confidence table ─────────────────────────────────────────────────────
    print(f"\n{'Model':<22} {'Predicted Class':<16} {'Confidence':>12}")
    print("-" * 52)
    for name, idx, probs in [
        ("PyTorch CNN",    pt_idx,  pt_probs),
        ("TensorFlow CNN", tf_idx,  tf_probs),
        ("VGG19 Transfer", vgg_idx, vgg_probs),
    ]:
        print(f"{name:<22} {CLASS_NAMES[idx]:<16} {probs[idx]:>12.4f}")


# Change NEW_IMAGE_PATH to your own image and run this cell
NEW_IMAGE_PATH = Path("new_image.jpg")   # <-- update this
classify_new_image(NEW_IMAGE_PATH)